# Chapter 33: Heat flow and finite differences

The code for this chapter from [*Matrices in Practice*](https://trgoodale.github.io/matrix-book/part-6/33-finite-differences.html), by Tom Goodale.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trgoodale/matrix-book/blob/main/code/python/33-finite-differences.ipynb)

Run the cells in order, from the top: later cells use variables set in earlier ones. Change the numbers and run them again to see what happens. The code needs NumPy, SciPy and Matplotlib (and pandas in a few chapters), all of which Colab already has.

Code licensed under the [MIT licence](https://trgoodale.github.io/matrix-book/LICENSE.md).

## In code

The Kronecker product builds the plate matrix in one line. The example solves a $49 \times 49$ grid, 2401 unknowns, with a sparse direct solver and with conjugate gradients.

In [ ]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import spsolve, cg

# Example 2: accuracy in one dimension
for n in [3, 7, 15, 31, 63]:
    h = 1 / (n + 1)
    x = np.arange(1, n + 1) * h
    A = sp.diags([-1.0, 2.0, -1.0], [-1, 0, 1], shape=(n, n), format="csc") / h**2
    u = spsolve(A, np.pi**2 * np.sin(np.pi * x))
    print(f"n = {n:2d}, h = {h:.5f}, error = {np.max(np.abs(u - np.sin(np.pi * x))):.1e}")

# A plate with a hot top edge
n = 49
T = sp.diags([-1.0, 2.0, -1.0], [-1, 0, 1], shape=(n, n))
I = sp.identity(n)
A = (sp.kron(I, T) + sp.kron(T, I)).tocsr()
b = np.zeros((n, n))
b[0, :] = 100                              # neighbours on the hot edge
u = spsolve(A, b.ravel())
print(A.shape, A.nnz, "non-zeros; centre =", u.reshape(n, n)[n // 2, n // 2])

steps = []
x, info = cg(A, b.ravel(), rtol=1e-8, callback=lambda xk: steps.append(1))
print("CG steps:", len(steps))

## Exercises

The code exercises from the [end of the chapter](https://trgoodale.github.io/matrix-book/part-6/33-finite-differences.html#sec-finite-differences-exercises). Worked solutions are in the notebook `solutions-part-6.ipynb`, and in the [book's solutions](https://trgoodale.github.io/matrix-book/appendices/solutions-6.html#sec-solutions-finite-differences).

### Exercise 6

Solve the plate problem with the top at 100 °C and the other edges at 0 °C for $n = 10, 20, 40, 80$, with a sparse direct solver and with CG. Record the time taken and the number of CG steps. How does the number of CG steps grow with $n$? Plot the temperature along the vertical centre line for $n = 80$.

In [ ]:
# Your code here